In [1]:
import pandas as pd
import numpy as np
import re
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
from nltk.stem import WordNetLemmatizer
df = pd.read_csv(r'D:\aiLearning\tuteDude_learning\assignment17\harry_potter_reviews.csv')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_preprocess(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercasing + Noise removal (from Task 3)
    text = text.lower()
    text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    # 2. Stopword removal
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    
    # 3. Lemmatization (preferred over stemming)
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(lemmatized)

# Apply final pipeline
df['final_clean_text'] = df['comment'].apply(nlp_preprocess)

In [3]:
import numpy as np

# Vocabulary
vocab = ['data', 'deep', 'enjoy', 'for', 'fun', 'great', 'i', 'is', 'learning', 'machine', 'of', 'programming', 'python', 'science', 'subset']
vocab_dict = {word: idx for idx, word in enumerate(vocab)}

sentences = [
    "I love machine learning",
    "Machine learning is fun",
    "Deep learning is a subset of machine learning",
    "I enjoy python programming",
    "Python is great for data science"
]

# Manual One-Hot Encoding
one_hot_matrix = []
for sentence in sentences:
    words = sentence.lower().split()
    vector = [0] * len(vocab)
    for word in words:
        if word in vocab_dict:          # ignore 'love' and 'a' as they are not in vocab
            vector[vocab_dict[word]] = 1
    one_hot_matrix.append(vector)

one_hot_matrix = np.array(one_hot_matrix)

print("Vocabulary:", vocab)
print("\nOne-Hot Encoded Matrix:")
print(one_hot_matrix)

Vocabulary: ['data', 'deep', 'enjoy', 'for', 'fun', 'great', 'i', 'is', 'learning', 'machine', 'of', 'programming', 'python', 'science', 'subset']

One-Hot Encoded Matrix:
[[0 0 0 0 0 0 1 0 1 1 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 1 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 1 1 1 1 0 0 0 1]
 [0 0 1 0 0 0 1 0 0 0 0 1 1 0 0]
 [1 0 0 1 0 1 0 1 0 0 0 0 1 1 0]]


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# Task 2: Using CountVectorizer with binary=True (One-Hot Encoding)
vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(sentences)

# Vocabulary
vocab = vectorizer.get_feature_names_out()
print("Vocabulary:", list(vocab))

# Encoded matrix as DataFrame for better readability
df = pd.DataFrame(X.toarray(), columns=vocab)
print("\nOne-Hot Encoded Matrix:")
print(df)

Vocabulary: ['data', 'deep', 'enjoy', 'for', 'fun', 'great', 'is', 'learning', 'love', 'machine', 'of', 'programming', 'python', 'science', 'subset']

One-Hot Encoded Matrix:
   data  deep  enjoy  for  fun  great  is  learning  love  machine  of  \
0     0     0      0    0    0      0   0         1     1        1   0   
1     0     0      0    0    1      0   1         1     0        1   0   
2     0     1      0    0    0      0   1         1     0        1   1   
3     0     0      1    0    0      0   0         0     0        0   0   
4     1     0      0    1    0      1   1         0     0        0   0   

   programming  python  science  subset  
0            0       0        0       0  
1            0       0        0       0  
2            0       0        0       1  
3            1       1        0       0  
4            0       1        1       0  


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

sentences = [
    "I love machine learning",
    "Machine learning is fun",
    "Deep learning is a subset of machine learning",
    "I enjoy python programming",
    "Python is great for data science"
]

# Task 3: Bag of Words using CountVectorizer (default = frequency count)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(sentences)

# Vocabulary
vocab = vectorizer.get_feature_names_out()
print("Vocabulary Size:", len(vocab))
print("Vocabulary:", list(vocab))

# BoW Matrix as DataFrame
bow_df = pd.DataFrame(X.toarray(), columns=vocab)
print("\nBag of Words Matrix:")
print(bow_df)

Vocabulary Size: 15
Vocabulary: ['data', 'deep', 'enjoy', 'for', 'fun', 'great', 'is', 'learning', 'love', 'machine', 'of', 'programming', 'python', 'science', 'subset']

Bag of Words Matrix:
   data  deep  enjoy  for  fun  great  is  learning  love  machine  of  \
0     0     0      0    0    0      0   0         1     1        1   0   
1     0     0      0    0    1      0   1         1     0        1   0   
2     0     1      0    0    0      0   1         2     0        1   1   
3     0     0      1    0    0      0   0         0     0        0   0   
4     1     0      0    1    0      1   1         0     0        0   0   

   programming  python  science  subset  
0            0       0        0       0  
1            0       0        0       0  
2            0       0        0       1  
3            1       1        0       0  
4            0       1        1       0  


In [6]:
import numpy as np

# Total frequency of each word across the entire dataset
word_frequencies = np.sum(X.toarray(), axis=0)

# Create frequency dictionary
freq_dict = dict(zip(vocab, word_frequencies))

# Sort by frequency (descending)
sorted_freq = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)

print("Word Frequencies:")
for word, freq in sorted_freq:
    print(f"{word:12} : {freq}")

Word Frequencies:
learning     : 4
is           : 3
machine      : 3
python       : 2
data         : 1
deep         : 1
enjoy        : 1
for          : 1
fun          : 1
great        : 1
love         : 1
of           : 1
programming  : 1
science      : 1
subset       : 1


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

sentences = [
    "I love machine learning",
    "Machine learning is fun",
    "Deep learning is a subset of machine learning",
    "I enjoy python programming",
    "Python is great for data science"
]

print("=== Task 5: N-grams Comparison ===\n")

# 1. Unigrams
vec_uni = CountVectorizer(ngram_range=(1, 1))
X_uni = vec_uni.fit_transform(sentences)
print(f"Unigrams  → Vocabulary Size: {len(vec_uni.get_feature_names_out())}")

# 2. Bigrams
vec_bi = CountVectorizer(ngram_range=(2, 2))
X_bi = vec_bi.fit_transform(sentences)
print(f"Bigrams   → Vocabulary Size: {len(vec_bi.get_feature_names_out())}")

# 3. Trigrams
vec_tri = CountVectorizer(ngram_range=(3, 3))
X_tri = vec_tri.fit_transform(sentences)
print(f"Trigrams  → Vocabulary Size: {len(vec_tri.get_feature_names_out())}")

print("\nSample Feature Vectors for First Sentence (truncated):")
print("Unigram  :", X_uni[0].toarray()[0][:15], "...")
print("Bigram   :", X_bi[0].toarray()[0][:10], "...")
print("Trigram  :", X_tri[0].toarray()[0][:8], "...")

# Show actual bigrams and trigrams
print("\nSome Bigrams:", sorted(vec_bi.get_feature_names_out())[:12])
print("Some Trigrams:", sorted(vec_tri.get_feature_names_out())[:8])

=== Task 5: N-grams Comparison ===

Unigrams  → Vocabulary Size: 15
Bigrams   → Vocabulary Size: 15
Trigrams  → Vocabulary Size: 13

Sample Feature Vectors for First Sentence (truncated):
Unigram  : [0 0 0 0 0 0 0 1 1 1 0 0 0 0 0] ...
Bigram   : [0 0 0 0 0 0 0 0 0 1] ...
Trigram  : [0 0 0 0 0 0 0 0] ...

Some Bigrams: ['data science', 'deep learning', 'enjoy python', 'for data', 'great for', 'is fun', 'is great', 'is subset', 'learning is', 'love machine', 'machine learning', 'of machine']
Some Trigrams: ['deep learning is', 'enjoy python programming', 'for data science', 'great for data', 'is great for', 'is subset of', 'learning is fun', 'learning is subset']


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Implementation
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(sentences)

print("TF-IDF Vocabulary Size:", len(tfidf_vectorizer.get_feature_names_out()))
print("TF-IDF Matrix Shape:", X_tfidf.shape)

# Show TF-IDF matrix for better understanding
tfidf_df = pd.DataFrame(X_tfidf.toarray(), 
                       columns=tfidf_vectorizer.get_feature_names_out())

print("\nTF-IDF Matrix (rounded to 3 decimals):")
print(tfidf_df.round(3))

TF-IDF Vocabulary Size: 15
TF-IDF Matrix Shape: (5, 15)

TF-IDF Matrix (rounded to 3 decimals):
    data   deep  enjoy    for    fun  great     is  learning   love  machine  \
0  0.000  0.000  0.000  0.000  0.000  0.000  0.000     0.486  0.726    0.486   
1  0.000  0.000  0.000  0.000  0.653  0.000  0.437     0.437  0.000    0.437   
2  0.000  0.419  0.000  0.000  0.000  0.000  0.281     0.561  0.000    0.281   
3  0.000  0.000  0.614  0.000  0.000  0.000  0.000     0.000  0.000    0.000   
4  0.443  0.000  0.000  0.443  0.000  0.443  0.297     0.000  0.000    0.000   

      of  programming  python  science  subset  
0  0.000        0.000   0.000    0.000   0.000  
1  0.000        0.000   0.000    0.000   0.000  
2  0.419        0.000   0.000    0.000   0.419  
3  0.000        0.614   0.496    0.000   0.000  
4  0.000        0.000   0.357    0.443   0.000  


In [12]:
print("=== Effect of Vectorizer Parameters on Vocabulary Size ===\n")

# 1. Default
vec_default = CountVectorizer()
print(f"1. Default                  → Vocabulary Size: {len(vec_default.fit(sentences).vocabulary_)}")

# 2. max_features (limits to top N most frequent features)
vec_maxf = CountVectorizer(max_features=10)
print(f"2. max_features=10          → Vocabulary Size: {len(vec_maxf.fit(sentences).vocabulary_)}")

# 3. min_df (ignore words that appear in fewer than N documents)
vec_mindf = CountVectorizer(min_df=2)
print(f"3. min_df=2                 → Vocabulary Size: {len(vec_mindf.fit(sentences).vocabulary_)}")

# 4. max_df (ignore words that appear in more than X% of documents)
vec_maxdf = CountVectorizer(max_df=0.8)   # ignore terms in >80% documents
print(f"4. max_df=0.8               → Vocabulary Size: {len(vec_maxdf.fit(sentences).vocabulary_)}")

# 5. Combined strict filtering
vec_strict = CountVectorizer(min_df=2, max_df=0.9, max_features=12)
print(f"5. min_df=2 + max_df=0.9 + max_features=12 → Vocabulary Size: {len(vec_strict.fit(sentences).vocabulary_)}")

=== Effect of Vectorizer Parameters on Vocabulary Size ===

1. Default                  → Vocabulary Size: 15
2. max_features=10          → Vocabulary Size: 10
3. min_df=2                 → Vocabulary Size: 4
4. max_df=0.8               → Vocabulary Size: 15
5. min_df=2 + max_df=0.9 + max_features=12 → Vocabulary Size: 4


Task 10: Conceptual Questions (Brief Answers)
1. Difference between One-Hot Encoding and BoW

One-Hot Encoding: Binary representation (0 or 1). Only tells whether a word is present in the sentence. Ignores frequency and order.
Bag of Words (BoW): Uses actual word counts (frequency). Captures how many times a word appears. Still ignores order.

→ One-Hot is a special case of BoW where counts are capped at 1 (binary=True).

2. Why N-grams increase dimensionality

N-grams create features from combinations of words instead of single words:

Unigrams: "machine", "learning"
Bigrams: "machine learning", "learning is", "deep learning"
Trigrams: "deep learning is", "machine learning is"

Each new combination becomes a separate feature → vocabulary size grows significantly (often exponentially with higher n). This captures local context but leads to higher dimensionality and more sparsity.

3. When to prefer TF-IDF over BoW

Prefer TF-IDF when:

You want to reduce the importance of very common words ("is", "learning", "the", etc.).
The dataset is large and contains many repetitive/stop words.
You need better feature quality for classification, clustering, or similarity tasks.
BoW is fine for very small datasets or when raw frequency is important.

TF-IDF = Term Frequency × Inverse Document Frequency (down-weights common terms).

4. Limitations of count-based vectorization (BoW / TF-IDF)


Ignores word order and semantics (except partially with N-grams).
High dimensionality and sparsity (curse of dimensionality).
Does not capture meaning or synonyms (e.g., "happy" and "joyful" are treated as completely different).
Sensitive to rare words and noise.
Poor performance on large vocabularies without proper parameter tuning.
Cannot handle out-of-vocabulary words in new test data well.

These are the main reasons modern NLP moved to word embeddings (Word2Vec, GloVe) and transformers (BERT, etc.).